In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_7 import BioJepa, BioJepaConfig
from training_v0_7 import create_model, maybe_compile
from config_v0_7 import DataConfig
from evals.evals import EvalContext, run_encoder_evals, run_composer_evals, run_ac_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/v0_7').expanduser()
ref_root = Path('~/data/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir=ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cuda


In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer= 6,
    heads= 4,
    embed_dim= 256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=2.38,
    film_linear_multiple=0.81,
    sim_coeff=50,
    std_coeff=25,
    cov_coeff=1,
    pert_latent_dim=128,
    pert_mode_dim=64,
)


EVAL_BATCH_SIZE = 32

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters()):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters()):,}')

Student/Teacher: 7,979,650
ACpredictor: 9,987,072
PerturbationComposer: 444,224


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_ac_final.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [6]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
pt_eval_results = run_encoder_evals(eval_ctx)

save_report(pt_eval_results, data_cfg.eval_results_dir / 'encoder_eval_report.json')
pt_eval_results

Using cuda
found 135 shards for split test


batch_invariance: Extracting embeddings: 100%|█████████| 10800/10800 [06:41<00:00, 26.93it/s]


Training classifiers...
batch_invariance: Batch=0.0521 (22.5x), Pert=0.0281 (31.1x)
batch_invariance summary: global_ratio=0.541, within_dataset_macro_ratio=0.462
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
gene_embedding_pathways: KEGG ratio=1.1854
essential_gene_prediction: Pearson=0.1228, AUROC=0.5797
found 135 shards for split test


cell_type_probing: Extracting embeddings: 100%|████████| 10800/10800 [06:33<00:00, 27.47it/s]


Training cell type classifier...
cell_type_probing: Accuracy=0.8519 (3.4x chance), Macro F1=0.5661
found 135 shards for split test


...

latent_space_health: Extracting embeddings: 100%|██████| 10800/10800 [06:30<00:00, 27.65it/s]


latent_space_health: Eff_dim_90=12/256, Mean_var=0.2705, Isotropy=0.000001


W0421 00:42:55.990000 310020 site-packages/torch/_inductor/scheduler.py:3820] [0/1] Layout conflict detected for buf1: template expects FixedLayout('cuda:0', torch.float32, size=[320000, 1], stride=[1, 1]) but layout is frozen to FixedLayout('cuda:0', torch.float32, size=[320000, 1], stride=[1, 320000])
W0421 00:42:56.003000 310020 site-packages/torch/_inductor/scheduler.py:3820] [0/1] Layout conflict detected for buf1: template expects FixedLayout('cuda:0', torch.float32, size=[320000, 1], stride=[1, 1]) but layout is frozen to FixedLayout('cuda:0', torch.float32, size=[320000, 1], stride=[1, 320000])
W0421 00:42:56.039000 310020 site-packages/torch/_inductor/scheduler.py:3820] [0/1] Layout conflict detected for buf1: template expects FixedLayout('cuda:0', torch.float32, size=[320000, 1], stride=[1, 1]) but layout is frozen to FixedLayout('cuda:0', torch.float32, size=[320000, 1], stride=[1, 320000])
W0421 00:42:56.098000 310020 site-packages/torch/_inductor/scheduler.py:3820] [0/1] L

permutation_invariance [adamson]: mean=0.486570, min=-0.760778
permutation_invariance [k562e_raw]: mean=0.457878, min=-0.801958
permutation_invariance [k562gw]: mean=0.446688, min=-0.924154
permutation_invariance [norman]: mean=0.521346, min=-0.947527
permutation_invariance [rep1e]: mean=0.443030, min=-0.926351
permutation_invariance [sciplex]: mean=0.411976, min=-0.962664
permutation_invariance overall: mean=0.462176, min=-0.962664
Saved report to /home/ubuntu/data/v0_7/eval_results/encoder_eval_report.json


{'batch_invariance': {'config': {'samples': 345600,
   'embedding_dim': 256,
   'num_batches': 432,
   'num_perturbations': 1106},
  'batch_classifier': {'accuracy': 0.05205439814814815,
   'chance': 0.0023148148148148147,
   'above_chance_ratio': 22.4875},
  'perturbation_classifier': {'accuracy': 0.02813946759259259,
   'chance': 0.0009041591320072332,
   'above_chance_ratio': 31.12225115740741},
  'invariance_ratio': 0.5405780989438577,
  'by_dataset': {'k562e_raw': {'config': {'samples': 46506,
     'embedding_dim': 256,
     'num_batches': 48,
     'num_perturbations': 286},
    'batch_classifier': {'accuracy': 0.08095033326166416,
     'chance': 0.020833333333333332,
     'above_chance_ratio': 3.88561599655988},
    'perturbation_classifier': {'accuracy': 0.014190496667383359,
     'chance': 0.0034965034965034965,
     'above_chance_ratio': 4.0584820468716405},
    'invariance_ratio': 0.1752988047808765},
   'k562gw': {'config': {'samples': 189095,
     'embedding_dim': 256,
    

In [7]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [8]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
align_eval_results = run_composer_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'composer_eval_report.json')
align_eval_results

Using cuda
Loaded 10791 v0.7 alignment pairs
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Encoded DNA sequences: torch.Size([11643, 128])
Encoded chemical sequences: torch.Size([188, 128])
Loaded target bank: torch.Size([9975, 320])
Encoded protein targets: torch.Size([9975, 128])
seq_to_target_retrieval: dna_mrr=0.0398
cross_modality_target_consistency: Within=0.5200, Between=0.4128, Ratio=1.26x
seq_target_gap_analysis: dna_gap=0.86
paired_alignment_quality: dna_sim=0.4768
mode_sensitivity: Classification_acc=0.4714 (3.3x chance)
mode_semantic_consistency: semantic_gap=-0.0549, cross_mode_mrr=0.0591
fusion_quality: Fused_var=0.6886, Seq_var=0.5353, Target_var=0.0846
missing_data_robustness: Fused_MRR=0.5749, Seq_only=0.0490, Target_only=0.9975
found 135 shards for split test


multi_pert_alignment: Scanning for multi-pert: 100%|██████| 500/500 [00:01<00:00, 251.55it/s]


Loaded 27264 HGNC gene family annotations
target_family_probing: seq_only=0.0851, target_only=0.3645, fused=0.2675
action_vector_pathways DNA: ratio=0.9983042644577172
Saved report to /home/ubuntu/data/v0_7/eval_results/composer_eval_report.json


{'seq_to_target_retrieval': {'config': {'n_targets': 9975},
  'by_modality': {'dna': {'mrr': 0.03979691471935104,
    'median_rank': 742.5,
    'mean_rank': 1763.9435559736594,
    'n_queries': 10630,
    'n_targets': 9975,
    'recall_at_k': {'1': 0.01730950141110066,
     '5': 0.05371589840075259,
     '10': 0.0794920037629351,
     '20': 0.11533396048918156,
     '50': 0.18071495766698026}}}},
 'cross_modality_target_consistency': {'config': {'n_valid_targets': 921,
   'n_within_pairs': 1408,
   'n_between_pairs': 5000},
  'metrics': {'within_target_sim': 0.5200058221817017,
   'between_target_sim': 0.41280980258882044,
   'consistency_ratio': 1.2596741136490257}},
 'seq_target_gap_analysis': {'target_variance': 11.23940658569336,
  'n_targets': 9975,
  'dna': {'seq_variance': 70.82473754882812,
   'centroid_distance': 4.724379539489746,
   'mean_within_seq': 11.852827309643214,
   'mean_seq_to_target': 10.168132781982422,
   'gap_ratio': 0.857865597494181,
   'n_sequences': 11643}}

In [9]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [10]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
keys = decoder.load_state_dict(decoder_sd)
keys

<All keys matched successfully>

In [11]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED,
})
full_eval_results = run_ac_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'ac_eval_report.json')
full_eval_results

Using cuda
found 135 shards for split test


Running test inference:   0%|                                      | 0/10800 [00:00<?, ?it/s]

Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


Running test inference:   1%|▏                          | 81/10800 [00:24<6:10:03,  2.07s/it]/home/ubuntu/code/biojepa/evals/evals.py:643: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_top, t_top)
Running test inference: 100%|██████████████████████████| 10800/10800 [45:26<00:00,  3.96it/s]


Aggregated 1416 single-pert, 13 multi-pert perturbations, 345600 samples, 135 shards
  adamson: 11 perturbations, 4288 samples
  k562e_raw: 286 perturbations, 46506 samples
  k562gw: 1053 perturbations, 189095 samples
  norman: 10 perturbations, 9472 samples
  rep1e: 287 perturbations, 22838 samples
  sciplex: 54 perturbations, 73401 samples
Cached test inference to /home/ubuntu/data/v0_7/test_inference_cache (135 shards)
expression_prediction: Pearson=0.6274, R2=0.2280, Centroid_acc=0.0008
gene_level_analysis: Dir_acc=0.2228, Top50_acc=0.2799


perturbation_retrieval (dna): 100%|██████████████████████| 200/200 [2:21:10<00:00, 42.35s/it]


perturbation_retrieval (dna): MRR=0.0005


perturbation_retrieval (chemical): 100%|█████████████████████| 54/54 [00:36<00:00,  1.47it/s]


perturbation_retrieval (chemical): MRR=0.0247
Loaded dataset gene masks: ['k562e_raw', 'rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']
uncertainty_calibration: ECE=0.1875, Monotonicity=55.56%
moa_matching expression: Within=0.9488, Between=0.9443, Gap=0.0045, Ratio=1.0048x
moa_matching latent: Within=0.9465, Between=0.9515, Gap=-0.0050, Ratio=0.9947x
Loaded Norman combo mapping: 132 combos
Loaded Norman single-gene deltas: 105 genes
Loaded Norman GI subtypes: 88 combos
Loaded dataset splits: ['k562e_raw', 'rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']
combination_perturbation: 13 combo perts, 4283 samples, 13 additive baseline, 5 GI-labeled, 13 generalization-classified
dose_response: monotonicity=48.77%, real_mono=48.77%, spearman=-0.0028, curve_sim=0.31779999250987184
Saved report to /home/ubuntu/data/v0_7/eval_results/ac_eval_report.json


{'expression_prediction': {'config': {'test_perturbations': 1416,
   'genes': 10000,
   'test_samples': 345600},
  'sample_level': {'mse': 1.4135693242880343,
   'pearson_r_top20': 0.5229911096357911},
  'perturbation_level': {'r2_all_genes': {'mean': 0.22804880487380055,
    'median': 0.2790887653827667},
   'r2_top50_degs': {'mean': -0.7616293221543737,
    'median': -0.6967307329177856},
   'mse': {'mean': 0.6891831159591675, 'median': 0.6682673692703247},
   'pearson_all_genes': {'mean': 0.6273545681125363,
    'median': 0.6408418416976929},
   'pearson_delta_all_genes': {'mean': 0.04971421452402433,
    'median': 0.04280279017984867},
   'pearson_top50_degs': {'mean': 0.13118902246709807,
    'median': 0.13556000590324402}},
  'centroid_accuracy': {'accuracy': 0.0007535795026375283, 'n_groups': 1327},
  'vs_baseline': {'beat_rate': 0.0, 'n_evaluated': 1416},
  'severity': {'pearson_r': 0.07700417935848236,
   'spearman_r': 0.3459312900204056},
  'error_by_magnitude': {'0-0.25': {'

In [12]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()